# Ver el predial en el formato que queremos

In [1]:
# -*- coding: utf-8 -*-
from pathlib import Path
import re
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt

# ========= Config =========
raw_folder = Path("Raw")
final_folder = Path("Final")

In [2]:
import geopandas as gpd
check  = gpd.read_parquet(raw_folder / "cdmx_predial_aprox.parquet") 

In [3]:
check.head()

,fid_attr,fid_2,alcaldia,colonia,calle_numero,codigo_postal,sup_terreno,valor_unitario_suelo,valor_suelo,valor_suelo_calc,...,anio_construccion,instal_esp,valor_constr_bruto,valor_constr_ajustado,valor_catastral_aprox,rango_tarifa,predial_bimestral_aprox,cve_vus,subsidio,geom
0,1.0,1966352.0,alvaro_obregon,cove,observatorio_297,1120_0,159.58,2604.93,415694.73,415694.73,...,2001.0,1.0,2244150.0,1903039.2,2318733.93,H,2200.74,a010513,644.260919,"MULTIPOLYGON (((-99.202 19.4014, -99.20196 19...."
1,2.0,1966365.0,alvaro_obregon,cove,observatorio_299,1120_0,171.81,2604.93,447553.02,447553.02,...,1969.0,0.0,2270340.0,1362204.0,1809757.02,F,1316.46,a010513,746.544515,"MULTIPOLYGON (((-99.20209 19.40139, -99.20206 ..."
2,3.0,1966359.0,alvaro_obregon,cove,poniente_79_67,1120_0,180.00,2604.93,468887.40,468887.40,...,1970.0,0.0,1269000.0,761400.0,1230287.40,E,620.58,a010513,643.418583,"MULTIPOLYGON (((-99.2021 19.40081, -99.20209 1..."
3,4.0,96480.0,alvaro_obregon,cove,observatorio_301,1120_0,176.31,2604.93,459275.21,459275.21,...,1970.0,1.0,257220.0,154332.0,613607.21,C,0.00,a010513,615.591443,"MULTIPOLYGON (((-99.20217 19.40138, -99.20209 ..."
4,5.0,1966363.0,alvaro_obregon,cove,observatorio_305,1120_0,200.00,2604.93,520986.00,520986.00,...,1997.0,0.0,3033000.0,2474928.0,2995914.00,I,3407.60,a010513,593.403316,"MULTIPOLYGON (((-99.20235 19.40132, -99.20235 ..."


Hacemos dos parquets, uno con geom y valor en suelo, otro con geom y predial aprox. 

In [6]:
# -*- coding: utf-8 -*-
from pathlib import Path
import geopandas as gpd
import pandas as pd

# ====== RUTAS ======
P_IN = Path(raw_folder) / "cdmx_predial_aprox.parquet"   # parquet origen
OUT_DIR = Path(final_folder)                              # carpeta de salida

EPSG_TARGET = 6372

def ensure_geom(g: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if "geom" in g.columns:
        g = g.set_geometry("geom", crs=g.crs)
    else:
        geom_cols = [c for c in g.columns if g[c].dtype.name == "geometry"]
        if not geom_cols:
            raise ValueError("No se detectó columna geométrica.")
        g = g.set_geometry(geom_cols[0], crs=g.crs).rename_geometry("geom")
    return g

def drop_z_vectorized(g: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    # Mucho más rápido que apply: convierte a WKB 2D y regresa a geometría
    wkb2d = g.geometry.to_wkb(output_dimension=2)
    g["geom"] = gpd.GeoSeries.from_wkb(wkb2d, crs=g.crs)
    return g.set_geometry("geom")

def to_epsg_6372_drop_z(g: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    g = ensure_geom(g)
    if g.crs is None:
        raise ValueError("El parquet no tiene CRS. Asigna uno con set_crs() antes de transformar.")
    g = drop_z_vectorized(g)
    if g.crs.to_epsg() != EPSG_TARGET:
        g = g.to_crs(EPSG_TARGET)
    return g.set_geometry("geom", crs=f"EPSG:{EPSG_TARGET}")

def make_layer(g: gpd.GeoDataFrame, source_col: str, out_name: str, var_label: str) -> Path:
    if source_col not in g.columns:
        raise KeyError(f"No existe la columna '{source_col}' en el parquet.")
    out = g[["geom", source_col]].copy()

    # columnas requeridas
    out["var"] = var_label
    out["valor"] = pd.to_numeric(out[source_col], errors="coerce")

    # limpia y ordena
    out = out.drop(columns=[source_col])
    out = out[~pd.isna(out["valor"])].reset_index(drop=True)
    out.insert(0, "id", range(1, len(out) + 1))
    out = out[["id", "var", "valor", "geom"]]

    # guarda
    p_out = OUT_DIR / out_name
    out.to_parquet(p_out, index=False)
    return p_out

# ====== RUN ======
g = gpd.read_parquet(P_IN)
g = to_epsg_6372_drop_z(g)

p1 = make_layer(g, "predial_bimestral_aprox", "predial_aprox.parquet", var_label="predial_aprox")
p2 = make_layer(g, "valor_suelo",            "valor_suelo.parquet",            var_label="valor_suelo")

print("Listo:")
print(" ->", p1)
print(" ->", p2)


Listo:
 -> Final\predial_aprox.parquet
 -> Final\valor_suelo.parquet


In [ ]:
check = gpd.read_parquet(final_folder/"predial_aprox.parquet")

In [8]:
check = gpd.read_parquet(final_folder/"valor_suelo.parquet")
check.head()

,id,var,valor,geom
0,1,valor_suelo,415694.73,"MULTIPOLYGON (((2793027.87 825472.578, 2793032..."
1,2,valor_suelo,447553.02,"MULTIPOLYGON (((2793018.656 825471.319, 279302..."
2,3,valor_suelo,468887.40,"MULTIPOLYGON (((2793018.742 825406.562, 279301..."
3,4,valor_suelo,459275.21,"MULTIPOLYGON (((2793009.668 825470.091, 279301..."
4,5,valor_suelo,520986.00,"MULTIPOLYGON (((2792991.804 825462.818, 279299..."
